<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-4_Agent_Harness_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 4. Gemini Agent Harness — 정상 종료까지 구현하기

**학습 목표:** `상태(State) → 모델의 도구 선택(Policy) → 도구 실행(Tool) → 관찰(Observation)`을 반복하는 Agent를 구현합니다.

**실행 방법:** 위에서 아래로 실행합니다. 키가 없으면 모의 LLM 테스트까지 모두 실행할 수 있습니다. 실제 Gemini 호출은 Colab 왼쪽 🔑 **비밀정보**에 `GEMINI_API_KEY`를 등록하고 노트북 액세스를 허용하면 실행됩니다.

**주의:** 검색 문서는 수업용 **가상 자료**로, 외부 인터넷 검색이 아닙니다. 이 노트북은 실제 LLM을 통한 *도구 선택*을 다룹니다.

## 1. 문서 데이터와 검색·계산 도구

검색 결과가 없으면 Agent가 검색어를 바꿔 재시도할 수 있습니다. 도구 자체는 LLM이 아니라 Python 함수입니다.

In [1]:
# 수업용 미니 문서 데이터베이스
DOCUMENTS = {
    "평가": "지능시스템 과목의 평가 비율: 과제 20%, 중간 30%, 기말 30%, 출석 20%.",
    "출석": "지능시스템 과목의 출석 반영 비율은 20%입니다.",
    "실습": "지능시스템 실습에서는 Workflow와 Agent의 제어 구조를 비교합니다."
}

# 검색에서 의미가 거의 없는 단어는 비교 대상에서 제외합니다.
STOPWORDS = {"지능시스템", "과목", "과목의", "대한", "몇", "몇퍼센트", "몇", "알려줘", "알려주세요", "은", "는", "이", "가"}

def search_notes(query: str) -> str:
    """키워드가 많이 겹치는 문서를 최대 2개 찾아 반환합니다."""
    import re

    # 1. 한글·영문·숫자 단위로 검색어를 분리합니다.
    words = re.findall(r"[가-힣A-Za-z0-9]+", query.lower())
    keywords = [word for word in words if word not in STOPWORDS]

    # 2. 모든 문서에 대해 관련도 점수를 계산합니다.
    matches = []
    for title, text in DOCUMENTS.items():
        content = (title + " " + text).lower()
        score = sum(1 for word in keywords if word in content)
        # 문서 제목 자체가 질문에 들어간 경우 조금 더 중요하게 봅니다.
        if title in query:
            score += 1
        if score > 0:
            matches.append((score, title, text))

    matches.sort(key=lambda x: x[0], reverse=True)
    if not matches:
        return "관련 문서를 찾지 못했습니다. '평가', '출석', '실습' 등의 짧은 키워드로 재검색해 보세요."

    # 문서 이름과 내용을 함께 전달하여 출처를 알 수 있게 합니다.
    return "\n".join(f"[{title}] {text}" for _, title, text in matches[:2])


def add_numbers(a: float, b: float) -> float:
    """두 수를 더하는 안전한 계산 도구입니다."""
    return float(a) + float(b)

# Agent가 실행할 수 있는 도구 목록(화이트리스트)
TOOLS = {"search_notes": search_notes, "add_numbers": add_numbers}

print("검색 테스트:", search_notes("지능시스템 과제 반영 비율"))
assert "과제 20%" in search_notes("지능시스템 과제 반영 비율")


검색 테스트: [평가] 지능시스템 과목의 평가 비율: 과제 20%, 중간 30%, 기말 30%, 출석 20%.
[출석] 지능시스템 과목의 출석 반영 비율은 20%입니다.


## 2. 모델 출력 정규화와 검증

LLM의 응답에 `action` 또는 `tool` 키가 올 수 있습니다. 또한 최종 답변이 `args.answer`에 올 수 있습니다. 이를 **하나의 실행 규격**으로 통일합니다. 다만 **알 수 없는 도구 이름은 절대 자동 허용하지 않습니다.**

In [2]:
import json
from numbers import Real

ALLOWED_ACTIONS = set(TOOLS) | {"finish"}


def normalize_action(raw):
    """모델 응답을 {'tool': ..., 'args': ..., 'answer': ...} 형태로 통일합니다."""
    # 문자열 JSON으로 들어왔다면 먼저 딕셔너리로 바꿉니다.
    if isinstance(raw, str):
        raw = json.loads(raw)
    if not isinstance(raw, dict):
        raise ValueError("모델 행동이 JSON 객체(dict)가 아닙니다.")

    action = dict(raw)  # 원본을 변경하지 않도록 복사합니다.

    # 'action' 키를 사용한 경우 'tool'로 통일합니다.
    if "tool" not in action and isinstance(action.get("action"), str):
        action["tool"] = action["action"]

    tool_name = action.get("tool")
    if not isinstance(tool_name, str) or tool_name not in ALLOWED_ACTIONS:
        raise ValueError(f"허용되지 않거나 빠진 tool: {tool_name!r}")

    args = action.get("args", {})
    if not isinstance(args, dict):
        raise ValueError("args는 JSON 객체여야 합니다.")

    # 도구별로 필요한 인자만 선택하고 타입을 확인합니다.
    if tool_name == "search_notes":
        query = args.get("query", action.get("query"))
        if not isinstance(query, str) or not query.strip():
            raise ValueError("search_notes에 비어 있지 않은 query가 필요합니다.")
        if len(query) > 200:
            raise ValueError("검색어는 200자를 넘을 수 없습니다.")
        return {"tool": tool_name, "args": {"query": query.strip()}}

    if tool_name == "add_numbers":
        a, b = args.get("a", action.get("a")), args.get("b", action.get("b"))
        if (not isinstance(a, Real) or isinstance(a, bool) or
                not isinstance(b, Real) or isinstance(b, bool)):
            raise ValueError("add_numbers에는 숫자 a와 b가 필요합니다.")
        return {"tool": tool_name, "args": {"a": a, "b": b}}

    # finish는 답변이 최상위 또는 args 내부에 있더라도 동일하게 처리합니다.
    answer = action.get("answer") or args.get("answer")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("finish에는 비어 있지 않은 answer가 필요합니다.")
    return {"tool": "finish", "args": {}, "answer": answer.strip()}


# 이전에 실제로 발생한 JSON 형식 오류를 테스트합니다.
assert normalize_action({"action": "search_notes", "args": {"query": "평가"}})["tool"] == "search_notes"
assert normalize_action({"tool": "finish", "args": {"answer": "과제는 20%"}})["answer"] == "과제는 20%"
assert normalize_action({"tool": "search_notes", "args": {"query": "평가"}})["args"]["query"] == "평가"
print("✅ 모델 출력 형식 정규화 테스트 통과")


✅ 모델 출력 형식 정규화 테스트 통과


## 3. Agent Harness: 상태 관리·도구 실행·종료

여기에서 **개발자는 허용된 행동과 종료 규칙을 정의**하지만, 실제 LLM 모드에서는 **모델이 다음 행동을 선택**합니다.

In [4]:
import copy


def run_agent(goal, policy, tools, max_steps=6):
    """Agent 실행 루프. (answer, log, status)를 반환합니다."""
    if not isinstance(max_steps, int) or max_steps < 1:
        raise ValueError("max_steps는 1 이상의 정수여야 합니다.")

    state = {"goal": goal, "observations": []}
    log = []

    for step in range(1, max_steps + 1):
        # ① LLM/API 호출 오류와 ② JSON 형식 오류를 구분합니다.
        try:
            raw = policy(copy.deepcopy(state))
        except Exception as error:
            # API 키, 모델명, 할당량 문제는 invalid_action이 아닙니다.
            log.append({"step": step, "error": f"{type(error).__name__}: {error}"[:300]})
            return None, log, "policy_error"

        try:
            # 모델이 반환한 행동을 검증/정규화합니다.
            action = normalize_action(raw)
        except (ValueError, TypeError, json.JSONDecodeError) as error:
            log.append({"step": step, "error": str(error), "raw_action": str(raw)[:300]})
            return None, log, "invalid_action"

        tool_name = action["tool"]
        args = action["args"]
        record = {"step": step, "tool": tool_name, "args": args}
        log.append(record)

        # 모델이 종료를 선택하면 답변을 반환합니다.
        if tool_name == "finish":
            record["answer"] = action["answer"]
            return action["answer"], log, "completed"

        # 등록되지 않은 도구는 실행하지 않습니다.
        if tool_name not in tools:
            record["error"] = "허용되지 않은 도구"
            return None, log, "invalid_tool"

        try:
            result = tools[tool_name](**args)
        except Exception as error:
            record["error"] = f"{type(error).__name__}: {error}"[:300]
            return None, log, "tool_error"

        # 결과를 기록하고, 다음 호출에서 모델이 관찰하게 합니다.
        observation = {"tool": tool_name, "args": args, "result": str(result)[:1000]}
        state["observations"].append(observation)
        record["result"] = observation["result"]

    return None, log, "max_steps"


def show_run(answer, log, status):
    print("상태:", status)
    print("답변:", answer)
    print("--- 실행 로그 ---")
    for row in log:
        print(row)


## 4. API 없이 Harness를 먼저 검증

이 단계는 인터넷 연결이나 API 키가 없어도 실행됩니다.

In [6]:
def test_policy(state):
    """수업용 모의 정책: 검색 → 답변. LLM 사용 예제는 아래에 있습니다."""
    if not state["observations"]:
        # 이전 오류 재현: tool 대신 action이라는 키를 반환
        return {"action": "search_notes", "args": {"query": "지능시스템 과제 반영 비율"}}
    # 이전 오류 재현: finish의 답변을 args 안에 넣음
    return {"tool": "finish", "args": {"answer": "지능시스템의 과제 반영 비율은 20%입니다."}}

mock_answer, mock_log, mock_status = run_agent(
    "지능시스템의 과제 반영 비율은?", test_policy, TOOLS
)
show_run(mock_answer, mock_log, mock_status)
assert mock_status == "completed" and "20%" in mock_answer
assert [row["tool"] for row in mock_log] == ["search_notes", "finish"]


상태: completed
답변: 지능시스템의 과제 반영 비율은 20%입니다.
--- 실행 로그 ---
{'step': 1, 'tool': 'search_notes', 'args': {'query': '지능시스템 과제 반영 비율'}, 'result': '[평가] 지능시스템 과목의 평가 비율: 과제 20%, 중간 30%, 기말 30%, 출석 20%.\n[출석] 지능시스템 과목의 출석 반영 비율은 20%입니다.'}
{'step': 2, 'tool': 'finish', 'args': {}, 'answer': '지능시스템의 과제 반영 비율은 20%입니다.'}


## 5. Harness의 안전장치 테스트

Agent가 끝없이 도구를 요청하거나, 허용되지 않은 도구를 선택할 때 안전하게 멈추는지 확인합니다.

In [7]:
# max_steps가 2라면 모델이 계속 검색해도 두 번까지만 실행합니다.
def always_search(state):
    return {"tool": "search_notes", "args": {"query": "평가"}}

_, limit_log, limit_status = run_agent("무한 검색", always_search, TOOLS, max_steps=2)
assert limit_status == "max_steps" and len(limit_log) == 2

# 허용 목록에 없는 파일 삭제 도구는 차단합니다.
def unsafe_policy(state):
    return {"tool": "delete_file", "args": {"path": "important.txt"}}

_, blocked_log, blocked_status = run_agent("파일 삭제", unsafe_policy, TOOLS)
assert blocked_status == "invalid_action"

print("최대 단계 제한:", limit_status)
print("허용되지 않은 도구 차단:", blocked_status)
print("✅ 안전장치 테스트 통과")

# 모델 자체에서 오류가 난 경우에는 invalid_action이 아니라 policy_error여야 합니다.
def broken_api_policy(state):
    raise RuntimeError("연결 오류 (모의 테스트)")

_, api_log, api_status = run_agent("모의 API 연결 오류", broken_api_policy, TOOLS)
assert api_status == "policy_error" and "연결 오류" in api_log[0]["error"]
print("API 오류와 JSON 형식 오류 구분:", api_status)


최대 단계 제한: max_steps
허용되지 않은 도구 차단: invalid_action
✅ 안전장치 테스트 통과
API 오류와 JSON 형식 오류 구분: policy_error


## 6. Google Gemini 연결 (Colab 비밀정보)

- <https://aistudio.google.com/apikey>에서 API 키를 발급합니다.
- Colab 왼쪽의 **비밀정보(🔑)** 에 이름 `GEMINI_API_KEY`로 등록하고 접근 권한을 켭니다.
- 모델명은 공식 모델 ID **`gemini-3.5-flash-lite`** 입니다.
- 키가 없는 경우 아래 **실제 API 실습만 건너뜁니다**. 기본/모의 테스트는 정상 실행됩니다.

> 모델 제공 범위, 요금 및 요청 한도는 계정과 시점에 따라 달라질 수 있습니다. 실제 API를 호출할 때 사용량이 발생할 수 있습니다.

In [8]:
import os

# Colab에서는 Secrets에 등록된 API 키를 읽습니다.
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or GEMINI_API_KEY
except (ImportError, KeyError):
    pass
except Exception:
    # 사용자가 권한을 허용하지 않았거나 등록하지 않은 경우
    pass

LIVE_ENABLED = bool(GEMINI_API_KEY)
MODEL_ID = "gemini-3.5-flash-lite"
print("Gemini 실제 실행:", "활성화" if LIVE_ENABLED else "API 키 없음 — 실제 호출 건너뜀")


Gemini 실제 실행: 활성화


## 7. 출력 JSON Schema를 지정하여 Gemini의 행동 형식 고정


In [9]:
# Google SDK 설치는 API 키가 있는 실제 실습에서만 필요합니다.
if LIVE_ENABLED:
    try:
        from google import genai
        from google.genai import types
    except ImportError:
        # Google Colab은 IPython 환경이므로 아래 명령으로 SDK를 설치할 수 있습니다.
        from IPython import get_ipython
        get_ipython().run_line_magic("pip", "-q install -U google-genai")
        from google import genai
        from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY)
else:
    client = None

# 가능한 행동 및 도구 인자 형태를 API에 명확히 전달합니다.
# 모델은 하나의 행동을 반환하며, 이후 다시 Harness가 값을 검증합니다.
ACTION_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "tool": {
            "type": "string",
            "enum": ["search_notes", "add_numbers", "finish"],
            "description": "다음에 실행할 단일 행동 이름"
        },
        "args": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "search_notes용 검색어"},
                "a": {"type": "number", "description": "add_numbers 첫 번째 숫자"},
                "b": {"type": "number", "description": "add_numbers 두 번째 숫자"},
                "answer": {"type": "string", "description": "finish용 최종 답변(내부 배치도 허용)"}
            }
        },
        "answer": {"type": "string", "description": "finish일 때 사용하는 최종 한국어 답변"}
    },
    "required": ["tool", "args"]
}

print("JSON Schema 준비 완료:", ACTION_JSON_SCHEMA["properties"]["tool"]["enum"])


JSON Schema 준비 완료: ['search_notes', 'add_numbers', 'finish']


## 8. LLM 정책: 관찰 결과를 보고 다음 행동 결정

형식 오류에는 **최대 1회 재시도**합니다. API 연결 오류·할당량 오류는 무리하게 재시도하지 않고 `policy_error`와 상세 메시지로 알려줍니다.

In [10]:
def gemini_policy(state):
    """Gemini 3.5 Flash-Lite가 관찰 상태를 보고 도구를 고릅니다."""
    if client is None:
        raise RuntimeError("Gemini API 키가 필요합니다.")

    prompt = f"""당신은 지능시스템 수업용 Agent의 다음 행동을 결정합니다.
목표: {state['goal']}
이전 관찰: {json.dumps(state['observations'], ensure_ascii=False)}

도구:
- search_notes: 문서에서 정보 검색 (args.query는 짧은 핵심 키워드)
- add_numbers: 숫자 두 개 더하기 (args.a, args.b)
- finish: 충분한 근거가 있으면 한국어 최종 답변 반환

반드시 JSON Schema 형식으로 행동 1개만 선택하세요.
1. 답을 위한 자료가 없다면 search_notes로 검색하세요.
2. 한번 검색이 실패하면 '평가', '출석', '실습' 같은 짧은 키워드로 바꿔 검색하세요.
3. 관찰 결과에 '과제 20%'가 있으면 바로 finish하고 '과제 반영 비율은 20%'라고 답하세요.
4. finish는 args를 빈 객체로 두고, 최상위 answer에 답변을 작성하세요.
5. 관찰에 없는 정보는 지어내지 마세요. 확인할 수 없으면 확인 불가라고 답하세요.
"""

    last_error = None
    for attempt in range(2):   # 형식 오류일 때만 1번 추가로 재요청합니다.
        retry_hint = "" if attempt == 0 else f"\n이전 출력 형식 오류: {last_error}. 반드시 schema에 맞춰 반환하세요."
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt + retry_hint,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_json_schema=ACTION_JSON_SCHEMA,
                temperature=0,
            ),
        )

        try:
            # response.text가 비어 있을 수도 있으므로 확인합니다.
            if not response.text:
                raise ValueError("Gemini가 빈 응답을 보냈습니다.")
            raw = json.loads(response.text)
            action = normalize_action(raw)
            print(f"Gemini 행동 결정: {action}")
            return action
        except (ValueError, TypeError, json.JSONDecodeError) as error:
            last_error = str(error)
            print(f"출력 형식 오류(시도 {attempt + 1}/2): {last_error}")

    raise ValueError("Gemini 행동 형식을 2회 시도 후에도 확인할 수 없습니다: " + str(last_error))


## 9. 실제 Gemini Agent 실행

원본 실습의 질문을 그대로 사용합니다. 정상적으로 검색 결과를 확보해 `finish`하면 `completed`와 근거 기반 답변이 반환됩니다. **라이브 LLM의 응답은 확정적이지 않으므로**, 실패 시 `error` 로그를 통해 원인을 확인합니다.

In [11]:
GOAL = "지능시스템 과목의 과제 반영 비율은 몇 퍼센트인가요?"

if LIVE_ENABLED:
    # 기존 셀보다 조금 넉넉하게 6단계까지 허용합니다.
    live_answer, live_log, live_status = run_agent(GOAL, gemini_policy, TOOLS, max_steps=6)
    show_run(live_answer, live_log, live_status)

    if live_status != "completed":
        print("⚠️ 정상 완료되지 않았습니다. 마지막 로그의 error 내용을 확인하세요.")
    elif "20%" not in live_answer:
        print("⚠️ 실행은 완료했으나, 답변의 정확도는 별도로 검증해야 합니다.")
else:
    print("Gemini API 키가 없어 실제 호출을 건너뜁니다.")
    print("아래 셀에서 API 없이 예측 가능한 모의 LLM 결과를 검증합니다.")


Gemini 행동 결정: {'tool': 'search_notes', 'args': {'query': '지능시스템'}}
Gemini 행동 결정: {'tool': 'search_notes', 'args': {'query': '평가'}}
Gemini 행동 결정: {'tool': 'finish', 'args': {}, 'answer': '지능시스템 과목의 과제 반영 비율은 20%입니다.'}
상태: completed
답변: 지능시스템 과목의 과제 반영 비율은 20%입니다.
--- 실행 로그 ---
{'step': 1, 'tool': 'search_notes', 'args': {'query': '지능시스템'}, 'result': "관련 문서를 찾지 못했습니다. '평가', '출석', '실습' 등의 짧은 키워드로 재검색해 보세요."}
{'step': 2, 'tool': 'search_notes', 'args': {'query': '평가'}, 'result': '[평가] 지능시스템 과목의 평가 비율: 과제 20%, 중간 30%, 기말 30%, 출석 20%.'}
{'step': 3, 'tool': 'finish', 'args': {}, 'answer': '지능시스템 과목의 과제 반영 비율은 20%입니다.'}
